# Construção de arquitetura Transformer com PyTorch

- **Repositório:** `deep-learning-neural-networks`  
- **Tecnologias:** PyTorch, TensorFlow  
- **Arquivo Original:** `Projeto_Transformador.ipynb`  

---


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import numpy as np
from PIL import Image, ImageEnhance
from itertools import product
#Aubuentatons

# =============================================
# CONFIGURAÇÕES
# =============================================
input_dir = '/content/drive/MyDrive/Cedulas/Reais'
output_dir = '/content/reais_todas_combinacoes'
os.makedirs(output_dir, exist_ok=True)

# =============================================
# PARÂMETROS DE COMBINAÇÕES
# =============================================
zooms       = [0.9, 1.0, 1.1]    # Zoom (3 opções)
brightnesses= [0.8, 1.0, 1.2]    # Brilho (3 opções)
contrasts   = [0.8, 1.0, 1.2]    # Contraste (3 opções)

# =============================================
# LOOP PRINCIPAL
# =============================================
for nota in os.listdir(input_dir):
    input_path  = os.path.join(input_dir, nota)
    output_path = os.path.join(output_dir, nota)
    os.makedirs(output_path, exist_ok=True)

    for img_name in os.listdir(input_path):
        img_path     = os.path.join(input_path, img_name)
        original_img = Image.open(img_path).convert('RGB').resize((256, 256))

        # Gera todas as combinações de zoom, brilho e contraste
        all_combinations = product(zooms, brightnesses, contrasts)

        for i, (zoom, bright, contrast) in enumerate(all_combinations):
            img = original_img.copy()

            # 1. Zoom
            if zoom != 1.0:
                w, h   = img.size
                new_w, new_h = int(w * zoom), int(h * zoom)
                img = img.resize((new_w, new_h))
                left = (new_w - 256) // 2
                top  = (new_h - 256) // 2
                img = img.crop((left, top, left + 256, top + 256))

            # 2. Brilho
            arr = np.array(img).astype(np.float32) * bright
            arr = np.clip(arr, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)

            # 3. Contraste
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(contrast)

            # Salva
            save_name = f"{os.path.splitext(img_name)[0]}_z{zoom}_b{bright}_c{contrast}.jpg"
            save_path = os.path.join(output_path, save_name)
            img.save(save_path)

print(f"✅ Todas as combinações (zoom, brilho, contraste) foram geradas em: {output_dir}")


✅ Todas as combinações (zoom, brilho, contraste) foram geradas em: /content/reais_todas_combinacoes


In [ ]:
# prompt: excluir pasta que tem arquivos dentro

import shutil
import os

# Caminho da pasta a ser excluída
pasta_para_excluir = '/content/reais_todas_combinacoes'

# Verifica se a pasta existe
if os.path.exists(pasta_para_excluir):
  try:
    shutil.rmtree(pasta_para_excluir)
    print(f"Pasta '{pasta_para_excluir}' e seu conteúdo excluídos com sucesso.")
  except OSError as e:
    print(f"Erro ao excluir a pasta '{pasta_para_excluir}': {e}")
else:
  print(f"A pasta '{pasta_para_excluir}' não existe.")


Pasta '/content/reais_todas_combinacoes' e seu conteúdo excluídos com sucesso.


In [ ]:
import os

# Caminho de entrada
input_dir = '/content/reais_todas_combinacoes'

# Dicionário para armazenar o número de imagens por valor de cédula
image_count_per_value = {}

# Contador total de imagens
total_images = 0

# Para cada pasta de valor (2, 5, 10 etc.)
for nota in os.listdir(input_dir):
    input_path = os.path.join(input_dir, nota)

    # Verificar se é uma pasta (representando um valor de cédula)
    if os.path.isdir(input_path):
        # Contar as imagens dentro da pasta
        image_count = len([img for img in os.listdir(input_path) if img.endswith(('.jpg', '.jpeg', '.png'))])

        # Armazenar a contagem de imagens para o valor da cédula
        image_count_per_value[nota] = image_count

        # Atualizar o total de imagens
        total_images += image_count

# Exibir a contagem de imagens por valor
print("Contagem de imagens por valor de cédula:")
for nota, count in image_count_per_value.items():
    print(f"Valor: {nota} - Imagens: {count}")

# Exibir o total de imagens
print(f"\nTotal de imagens: {total_images}")


Contagem de imagens por valor de cédula:
Valor: 200 - Imagens: 108
Valor: 2 - Imagens: 162
Valor: 100 - Imagens: 189
Valor: 5 - Imagens: 108
Valor: 50 - Imagens: 189
Valor: 20 - Imagens: 108
Valor: 10 - Imagens: 108

Total de imagens: 972


In [ ]:
pip install tensorflow matplotlib numpy pillow

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Parâmetros
dataset_dir = '/content/reais_todas_combinacoes'
img_size = 64
batch_size = 64
latent_dim = 100
epochs = 5000
save_interval = 10  # Salvar imagens a cada 10 épocas

# Função para carregar e preprocessar o dataset
def load_dataset(path_glob, img_size, batch_size):
    files = glob.glob(path_glob, recursive=True)
    def preprocess(f):
        img = Image.open(f).convert('RGB').resize((img_size, img_size))
        arr = (np.array(img).astype('float32') / 127.5) - 1.0  # Normalização para [-1, 1]
        return arr
    data = np.stack([preprocess(f) for f in files], axis=0)
    dataset = tf.data.Dataset.from_tensor_slices(data)
    dataset = dataset.shuffle(1024).batch(batch_size, drop_remainder=True)
    return dataset

dataset = load_dataset(dataset_dir + '/**/*.jpg', img_size, batch_size)

def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*256, use_bias=False, input_shape=(latent_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))  # (8, 8, 256)

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))  # (16, 16, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))   # (32, 32, 64)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))  # (64, 64, 3)

    return model


# Função para criar o discriminador
def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     input_shape=[img_size, img_size, 3]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

# Inicialização dos modelos
generator = make_generator_model()
discriminator = make_discriminator_model()

# Otimizadores
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

# Função de perda
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# Funções de perda para o discriminador e gerador
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Preparação para salvar imagens
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    predictions = (predictions + 1) / 2.0  # Desnormalização para [0, 1]

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i])
        plt.axis('off')

    if not os.path.exists('generated_images'):
        os.makedirs('generated_images')
    plt.savefig(f'generated_images/image_at_epoch_{epoch:04d}.png')
    plt.close()

# Vetor de entrada fixo para geração de imagens
seed = tf.random.normal([16, latent_dim])

# Função de treino
@tf.function
def train_step(images):
    noise = tf.random.normal([batch_size, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

# Loop de treino
for epoch in range(1, epochs + 1):
    for image_batch in dataset:
        g_loss, d_loss = train_step(image_batch)

    print(f'Época {epoch}, Perda do Gerador: {g_loss:.4f}, Perda do Discriminador: {d_loss:.4f}')

    if epoch % save_interval == 0:
        generate_and_save_images(generator, epoch, seed)

print("✅ Treinamento concluído. Use `generator.predict()` para gerar mais imagens.")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Época 1, Perda do Gerador: 0.8696, Perda do Discriminador: 0.7982
Época 2, Perda do Gerador: 0.9323, Perda do Discriminador: 0.9889
Época 3, Perda do Gerador: 1.3542, Perda do Discriminador: 0.5810
Época 4, Perda do Gerador: 2.2064, Perda do Discriminador: 0.2613
Época 5, Perda do Gerador: 1.8358, Perda do Discriminador: 0.3173
Época 6, Perda do Gerador: 2.4645, Perda do Discriminador: 0.1665
Época 7, Perda do Gerador: 2.6307, Perda do Discriminador: 0.1950
Época 8, Perda do Gerador: 3.1785, Perda do Discriminador: 0.1714
Época 9, Perda do Gerador: 2.8579, Perda do Discriminador: 0.1765
Época 10, Perda do Gerador: 3.4275, Perda do Discriminador: 0.1664
Época 11, Perda do Gerador: 4.2455, Perda do Discriminador: 0.1508
Época 12, Perda do Gerador: 3.9475, Perda do Discriminador: 0.3597
Época 13, Perda do Gerador: 2.7170, Perda do Discriminador: 0.4497
Época 14, Perda do Gerador: 1.6608, Perda do Discriminador: 0.5834
Época 15, Perda do Gerador: 2.2383, Perda do Discriminador: 0.3284
Époc

In [1]:
%cd /content


/content


In [2]:
!pip install --upgrade pip


In [3]:
pip install torch torchvision


In [5]:
!pip install ninja imageio-ffmpeg pyspng


In [6]:
!git clone https://github.com/NVlabs/stylegan3.git
%cd stylegan3


fatal: destination path 'stylegan3' already exists and is not an empty directory.
/content/stylegan3


In [9]:
!find /content/reais_todas_combinacoes -type f | wc -l   # Exibe total de arquivos .jpg/.png :contentReference[oaicite:5]{index=5}


972


In [8]:
# Garante que você está em /content, não em uma pasta removida
%cd /content


/content


In [1]:
# Desinstala qualquer NumPy residual 2.x
!pip uninstall -y numpy

# Instala NumPy 1.x compatível
!pip install numpy==1.23.5


Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5
  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.1 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.16.1 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
bigframes 2.1.0 requires numpy>=1.24.0, but you

In [3]:
%cd /content
!pip uninstall -y jax jaxlib


/content
Found existing installation: jax 0.5.1
Uninstalling jax-0.5.1:
  Successfully uninstalled jax-0.5.1
Found existing installation: jaxlib 0.5.1
Uninstalling jaxlib-0.5.1:
  Successfully uninstalled jaxlib-0.5.1


In [4]:
!pip install --upgrade tensorboard


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 54.7 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.16.2
    Uninstalling tensorboard-2.16.2:
      Successfully uninstalled tensorboard-2.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow 2.16.1 requires tensorboard<2.17,>=2.16, but you have tensorboard 2.19.0 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.16.1 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.16.1 which is incompatible.
tf-keras

In [5]:
# In a Colab code cell
import importlib, sys
print("jax:", importlib.util.find_spec("jax"))
print("jaxlib:", importlib.util.find_spec("jaxlib"))
import torch.utils.tensorboard as tb
print("TensorBoard writer available:", hasattr(tb, "SummaryWriter"))


jax: None
jaxlib: None
TensorBoard writer available: True


In [ ]:
%cd /content/stylegan3

!python train.py \
  --outdir=/content/stylegan3_runs/reais \
  --data=/content/reais_todas_combinacoes \
  --gpus=1 \
  --batch=16 \
  --cfg=stylegan3-t \
  --gamma=8.2 \
  --snap=50 \
  --metrics=none \
  --kimg=5000



/content/stylegan3

Training options:
{
  "G_kwargs": {
    "class_name": "training.networks_stylegan3.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 2
    },
    "channel_base": 32768,
    "channel_max": 512,
    "magnitude_ema_beta": 0.9994456359721023
  },
  "D_kwargs": {
    "class_name": "training.networks_stylegan2.Discriminator",
    "block_kwargs": {
      "freeze_layers": 0
    },
    "mapping_kwargs": {},
    "epilogue_kwargs": {
      "mbstd_group_size": 4
    },
    "channel_base": 32768,
    "channel_max": 512
  },
  "G_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.0025
  },
  "D_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.002
  },
  "loss_kwargs": {
    "class_name": "training.loss.StyleGAN2Loss",
    "r1_gamma": 8.2
  },
  "data_loader_kwargs": {
    "pin_memory": true,